# Reversing the force on a leaky dielectric sphere
Place a small lossy sphere in a weakly nonuniform alternating electric field. Change the particle conductivity and decide whether high- and low-frequency forces point toward the same region.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

particle_er, medium_er = 60., 10.
particle_sigma, medium_sigma = .002, .010    # S/m
radius_um = 5.
grad_rms_squared = 1e12                    # V²/m³
assert min(particle_er,medium_er,particle_sigma,medium_sigma,radius_um)>0

## Match total normal current and potential
The complex permittivity includes conduction for an exp(iωt) convention. The electric field is quoted as an rms amplitude. Keeping this convention avoids a factor of two in the time-averaged force.

In [ ]:
eps0=8.8541878128e-12; ep,em=particle_er*eps0,medium_er*eps0
low=(particle_sigma-medium_sigma)/(particle_sigma+2*medium_sigma)
high=(ep-em)/(ep+2*em)
tau=(ep+2*em)/(particle_sigma+2*medium_sigma)
fc=np.sqrt(-low/high)/(2*np.pi*tau) if low*high<0 else np.nan
freq=np.geomspace(.005/(2*np.pi*tau),200/(2*np.pi*tau),240)
omega=2*np.pi*freq
K=((ep-em)-1j*(particle_sigma-medium_sigma)/omega)/((ep+2*em)-1j*(particle_sigma+2*medium_sigma)/omega)
F=2*np.pi*em*(radius_um*1e-6)**3*grad_rms_squared*K.real
print("Low/high contrast",low,high,". Crossover frequency (Hz)",fc)
assert np.allclose(K.real,(low+high*(omega*tau)**2)/(1+(omega*tau)**2))

## Inspect a zero against both limiting contrasts
A zero in the real part erases the leading gradient force even though the imaginary part and dissipation can remain finite. Change the medium conductivity until the root disappears.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3.9),layout="constrained")
ax[0].semilogx(freq/1e6,K.real,label="Real part",color="#175868");ax[0].semilogx(freq/1e6,K.imag,label="Imaginary part",color="#a23d21",ls="--")
ax[0].set(xlabel="Frequency (MHz)",ylabel="Dipole contrast",title="Electrical response");ax[0].legend(fontsize=9)
ax[1].semilogx(freq/1e6,F*1e15,color="#375b9a");ax[1].set(xlabel="Frequency (MHz)",ylabel="Force along positive gradient (fN)",title="Time-averaged gradient force")
for a in ax:
    a.axhline(0,color="grey",lw=.8)
    if np.isfinite(fc): a.axvline(fc/1e6,color="grey",ls=":")
plt.show()